### Data Scientist Job Market Analysis - Glassdoor

The analysis helps job seekers understand market conditions, salary expectations, and identify optimal job search strategies.

In [4]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import re
import math
from statistics import mean
from pathlib import Path

### Setup and Dependencies

This will enable us to collect and analyze job posting data efficiently.

In [5]:
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()), options=options
)

url = "https://www.glassdoor.com/Job/jobs.htm?sc.keyword=data%20scientist&locT=&locId=&jobType="
driver.get(url)
time.sleep(8)
for _ in range(3):
    driver.execute_script("window.scrollBy(0, 1000);")
    time.sleep(3)

jobs = driver.find_elements("css selector", "li[data-test='jobListing']")
print(f"👉 Found {len(jobs)} job cards")

all_jobs = []

for job in jobs:
    try:
        title_elem = job.find_element("css selector", "a[data-test='job-title']")
        job_title = title_elem.text.strip()
        job_url = title_elem.get_attribute("href")
    except:
        job_title, job_url = None, None

    try:
        company = job.find_element(
            "css selector", "span.EmployerProfile_compactEmployerName__9MGcV"
        ).text.strip()
    except:
        company = None

    try:
        rating = job.find_element(
            "css selector", "span.rating-single-star_RatingText__5fdjN"
        ).text.strip()
    except:
        rating = None

    try:
        location = job.find_element(
            "css selector", "div[data-test='emp-location']"
        ).text.strip()
    except:
        location = None

    try:
        salary = job.find_element(
            "css selector", "div[data-test='detailSalary']"
        ).text.strip()
    except:
        salary = None

    try:
        posted_date = job.find_element(
            "css selector", "div[data-test='job-age']"
        ).text.strip()
    except:
        posted_date = None

    try:
        easy_apply = job.find_element(
            "css selector", "div.text-with-icon_LabelContainer__hmHB4"
        ).text.strip()
        easy_apply = "Yes" if "Easy Apply" in easy_apply else "No"
    except:
        easy_apply = "No"

    all_jobs.append(
        {
            "Job Title": job_title,
            "Job URL": job_url,
            "Company": company,
            "Location": location,
            "Salary": salary,
            "Company Rating": rating,
            "Posted Date": posted_date,
            "Easy Apply": easy_apply,
        }
    )

driver.quit()

df = pd.DataFrame(all_jobs)
print(df.head())
df.to_csv("glassdoor_job.csv", index=False, encoding="utf-8-sig")

print("\nScraping Completed! Data saved to glassdoor_job.csv")

👉 Found 30 job cards
                                 Job Title  \
0  Data Scientist, Innovation Lab - Remote   
1                           Data Scientist   
2                           Data Scientist   
3                    US LBM Data Scientist   
4                Data Scientist, Marketing   

                                             Job URL             Company  \
0  https://www.glassdoor.com/job-listing/data-sci...            Experian   
1  https://www.glassdoor.com/job-listing/data-sci...  Oland Technologies   
2  https://www.glassdoor.com/job-listing/data-sci...      Parkar Digital   
3  https://www.glassdoor.com/job-listing/us-lbm-d...     US LBM Holdings   
4  https://www.glassdoor.com/job-listing/data-sci...           Confluent   

        Location                                         Salary  \
0  United States                                           None   
1  Dade City, FL  $25.00 - $100.00 Per Hour (Employer provided)   
2         Remote                            

### Data Collection from Glassdoor

This section performs automated web scraping of Data Scientist job listings and saves the collected data to a CSV file for further analysis

In [6]:
in_data = "glassdoor_job.csv"


def parse_salary(s):

    if not isinstance(s, str) or not s.strip():
        return None, None

    text = s.strip()
    # Remove parenthetical notes
    text = re.sub(r"\(.*?\)", "", text)
    text = text.replace(",", "")
    text = text.strip()

    text = text.replace("per year", "").replace("/yr", "")

    def tok_to_int(tok):
        tok = tok.strip().replace("$", "").replace("Rs", "").strip()
        if tok == "":
            return None
        m = re.match(r"^([0-9]+(?:\.[0-9]+)?)\s*([kKmM])?$", tok)
        if m:
            val = float(m.group(1))
            unit = m.group(2)
            if unit:
                if unit.lower() == "k":
                    val = val * 1000
                elif unit.lower() == "m":
                    val = val * 1_000_000
            return int(round(val))
        try:
            return int(float(tok))
        except:
            return None

    # Range with dash or 'to'
    rng_match = re.search(r"([^\s\-–—]+)\s*(?:-|–|—|to)\s*([^\s\-–—]+)", text)
    if rng_match:
        a = tok_to_int(rng_match.group(1))
        b = tok_to_int(rng_match.group(2))
        return a, b

    # "Up to X" or "Up to $120K"
    up_to = re.search(r"up to\s+([^\s]+)", text, flags=re.I)
    if up_to:
        val = tok_to_int(up_to.group(1))
        return None, val

    # "From X" or "Starting at X"
    from_match = re.search(r"(?:from|starting at)\s+([^\s]+)", text, flags=re.I)
    if from_match:
        val = tok_to_int(from_match.group(1))
        return val, None

    # Single number with '+' like "$120K+"
    plus_match = re.match(r"^([^\s\+]+)\+$", text)
    if plus_match:
        val = tok_to_int(plus_match.group(1))
        return val, None

    # Single token numeric
    single = tok_to_int(text.split()[0])
    if single:
        return single, single

    return None, None


if not Path(in_data).exists():
    raise FileNotFoundError(f"Input CSV not found: {in_data}")

df = pd.read_csv(in_data)

# Make safe copies of key columns if names vary
salary_col_candidates = [c for c in df.columns if "salary" in c.lower()]
salary_col = salary_col_candidates[0] if salary_col_candidates else None
if salary_col is None:
    print(
        "No salary column found in CSV. Add a column containing salary text (e.g. 'Salary')."
    )
    df["SalaryRaw"] = None
else:
    df["SalaryRaw"] = df[salary_col].astype(str)

# Normalize Easy Apply column if present
easy_candidates = [c for c in df.columns if "easy" in c.lower()]
if easy_candidates:
    df["EasyApplyFlag"] = (
        df[easy_candidates[0]]
        .astype(str)
        .str.lower()
        .map(lambda x: "yes" if "y" in x or "easy" in x else "no")
    )
else:
    df["EasyApplyFlag"] = (
        df.get("Easy Apply", "")
        .astype(str)
        .str.lower()
        .map(lambda x: "yes" if "y" in x or "easy" in x else "no")
    )

# Company rating column
rating_candidates = [c for c in df.columns if "rating" in c.lower()]
rating_col = rating_candidates[0] if rating_candidates else None

# Compute salary_low, salary_high, salary_mid
salary_low = []
salary_high = []
salary_mid = []

for raw in df["SalaryRaw"].fillna("").tolist():
    low, high = parse_salary(raw)
    salary_low.append(low)
    salary_high.append(high)
    mid = None
    if low and high:
        mid = int(round((low + high) / 2))
    elif low:
        mid = low
    elif high:
        mid = high
    salary_mid.append(mid)

df["salary_low"] = salary_low
df["salary_high"] = salary_high
df["salary_mid"] = salary_mid

# Cast rating to numeric if available
if rating_col:
    df["rating_num"] = pd.to_numeric(df[rating_col], errors="coerce")
else:
    df["rating_num"] = pd.NA

# Clean company name
company_col_candidates = [
    c for c in df.columns if "company" in c.lower() or "employer" in c.lower()
]
company_col = company_col_candidates[0] if company_col_candidates else None
if company_col:
    df["company_clean"] = df[company_col].astype(str).str.strip()
else:
    # fallback lookups
    df["company_clean"] = (
        df.get(
            "Company",
            df.get("Company Name", df.get("company", pd.Series(["Unknown"] * len(df)))),
        )
        .astype(str)
        .str.strip()
    )

# Clean location
loc_candidates = [c for c in df.columns if "location" in c.lower()]
loc_col = loc_candidates[0] if loc_candidates else None
if loc_col:
    df["location_clean"] = df[loc_col].astype(str).str.strip()
else:
    df["location_clean"] = df.get("Location", pd.Series([None] * len(df))).astype(str)

# Clean EasyApplyFlag to Yes/No
df["EasyApplyFlag"] = (
    df["EasyApplyFlag"]
    .fillna("no")
    .apply(lambda x: "Yes" if str(x).lower().startswith("y") else "No")
)

### Data Preprocessing and Cleaning

This section handles the complex task of cleaning and standardizing the raw job data.
The clean data will enable more accurate and meaningful analysis in the next section.

In [7]:
# 1. Salary Insights

# Highest-paying and lowest-paying roles by salary_mid (ignore nulls)
df_salary = df.dropna(subset=["salary_mid"]).copy()
if df_salary.empty:
    print("No parsable salaries found. Exiting salary-based analysis.")
else:
    idx_max = df_salary["salary_mid"].idxmax()
    idx_min = df_salary["salary_mid"].idxmin()
    highest_role = df_salary.loc[idx_max]
    lowest_role = df_salary.loc[idx_min]

    print("\n1) SALARY INSIGHTS")
    print("Highest-paying role:")
    print(
        f"  Job Title: {highest_role.get('Job Title', highest_role.get('Name', 'N/A'))}"
    )
    print(f"  Company: {highest_role.get('company_clean')}")
    print(f"  Salary (mid): {highest_role['salary_mid']:,}")
    print(
        f"  Salary low/high: {highest_role['salary_low']:,} / {highest_role['salary_high']:,}"
    )

    print("\nLowest-paying role:")
    print(
        f"  Job Title: {lowest_role.get('Job Title', lowest_role.get('Name', 'N/A'))}"
    )
    print(f"  Company: {lowest_role.get('company_clean')}")
    print(f"  Salary (mid): {lowest_role['salary_mid']:,}")
    print(
        f"  Salary low/high: {lowest_role['salary_low']:,} / {lowest_role['salary_high']:,}"
    )

    avg_low = df_salary["salary_low"].dropna().mean()
    avg_high = df_salary["salary_high"].dropna().mean()
    avg_mid = df_salary["salary_mid"].mean()

    print(f"\nAverage salary low across jobs: {avg_low:,.0f}")
    print(f"Average salary high across jobs: {avg_high:,.0f}")
    print(f"Average salary midpoint across jobs: {avg_mid:,.0f}")


# 2. Company Analysis

print("\n2) COMPANY ANALYSIS")
unique_companies = df["company_clean"].dropna().unique()
print(f"Number of unique companies hiring: {len(unique_companies)}")

top3 = df["company_clean"].value_counts().head(3)
print("\nTop 3 companies by number of job postings:")
print(top3.to_string())


# 3. Easy Apply Advantage

print("\n3) EASY APPLY ADVANTAGE")
total_jobs = len(df)
easy_count = (df["EasyApplyFlag"] == "Yes").sum()
pct_easy = 100 * easy_count / total_jobs if total_jobs > 0 else 0
print(
    f"Percentage of jobs with Easy Apply: {pct_easy:.1f}% ({easy_count}/{total_jobs})"
)

easy_avg = df.loc[df["EasyApplyFlag"] == "Yes", "salary_mid"].dropna()
non_easy_avg = df.loc[df["EasyApplyFlag"] == "No", "salary_mid"].dropna()
if not easy_avg.empty:
    print(
        f"Average salary (mid) for Easy Apply jobs: Rs {easy_avg.mean():,.0f} (n={len(easy_avg)})"
    )
else:
    print("No salary data for Easy Apply jobs.")

if not non_easy_avg.empty:
    print(
        f"Average salary (mid) for Non-Easy Apply jobs: Rs {non_easy_avg.mean():,.0f} (n={len(non_easy_avg)})"
    )
else:
    print("No salary data for Non-Easy Apply jobs.")


# 4. Company Rating & Salary Relationship

print("\n4) COMPANY RATING vs SALARY")
both = df.dropna(subset=["rating_num", "salary_mid"])
if both.empty:
    print("Not enough data with both rating and salary to compare.")
else:
    # correlation
    corr = both["rating_num"].corr(both["salary_mid"])
    print(f"Correlation (rating vs salary_mid): {corr:.3f}")

    def rating_bucket(r):
        if r < 3.5:
            return "<3.5"
        if r < 4.0:
            return "3.5-3.99"
        return "4.0+"

    both = both.copy()
    both["rating_bucket"] = both["rating_num"].apply(rating_bucket)
    bucket_stats = (
        both.groupby("rating_bucket")["salary_mid"].agg(["count", "mean"]).sort_index()
    )
    print("\nAverage salary by rating bucket:")
    print(bucket_stats.to_string(formatters={"mean": "{:,.0f}".format}))


# 5. Location Trends

print("\n5) LOCATION TRENDS")
loc_counts = df["location_clean"].fillna("Unknown").value_counts().head(10)
print("Most common job locations (top 10):")
print(loc_counts.to_string())

# Compare major tech hubs
tech_hubs = [
    "San Francisco",
    "San Jose",
    "New York",
    "Seattle",
    "Washington",
    "Boston",
    "Austin",
    "Los Angeles",
]


def is_tech_hub(loc):
    if pd.isna(loc):
        return False
    for hub in tech_hubs:
        if hub.lower() in str(loc).lower():
            return True
    return False


df["is_tech_hub"] = df["location_clean"].apply(is_tech_hub)
tech = df[df["is_tech_hub"] & df["salary_mid"].notna()]["salary_mid"]
non_tech = df[~df["is_tech_hub"] & df["salary_mid"].notna()]["salary_mid"]

if not tech.empty:
    print(f"\nAverage salary in tech hubs (n={len(tech)}): Rs {tech.mean():,.0f}")
else:
    print("\nNo salary data for tech hub locations found in CSV.")

if not non_tech.empty:
    print(
        f"Average salary outside tech hubs (n={len(non_tech)}): Rs {non_tech.mean():,.0f}"
    )
else:
    print("No salary data outside tech hubs.")

if not tech.empty and not non_tech.empty:
    diff = tech.mean() - non_tech.mean()
    pct_diff = 100 * diff / non_tech.mean() if non_tech.mean() != 0 else float("nan")
    print(
        f"\nTech hubs pay on average Rs {diff:,.0f} more ({pct_diff:.1f}% higher) than non-tech locations."
    )


1) SALARY INSIGHTS
Highest-paying role:
  Job Title: Data Scientist, Marketing
  Company: Confluent
  Salary (mid): 174,000.0
  Salary low/high: 160,000.0 / 188,000.0

Lowest-paying role:
  Job Title: Data Scientist
  Company: ProntoDigital LLC
  Salary (mid): 50.0
  Salary low/high: 50.0 / 50.0

Average salary low across jobs: 73,015
Average salary high across jobs: 98,158
Average salary midpoint across jobs: 85,587

2) COMPANY ANALYSIS
Number of unique companies hiring: 29

Top 3 companies by number of job postings:
company_clean
Experian              2
Oland Technologies    1
Parkar Digital        1

3) EASY APPLY ADVANTAGE
Percentage of jobs with Easy Apply: 83.3% (25/30)
Average salary (mid) for Easy Apply jobs: Rs 77,524 (n=17)
Average salary (mid) for Non-Easy Apply jobs: Rs 113,000 (n=5)

4) COMPANY RATING vs SALARY
Correlation (rating vs salary_mid): 0.796

Average salary by rating bucket:
               count    mean
rating_bucket               
3.5-3.99           5 131,500
